# 🚗 Car Price Prediction with Machine Learning — CodeAlpha Data Science Internship (Task 3)

**Problem statement:** Predict the selling price of a used car from its characteristics (age, original showroom price, kilometres driven, fuel type, seller type, transmission, number of previous owners).

**Type of problem:** Supervised learning → **regression** (the answer is a number: a price).

**Dataset:** "Vehicle dataset" from CarDekho on Kaggle — https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho — use the file **`car data.csv`** (≈301 cars). Prices are in **lakhs of rupees** (1 lakh = ₹100,000).

**How to run:** download `car data.csv`, upload it next to this notebook (or into `data/`), then run all cells top to bottom.

## Step 1 — Import libraries
New tools compared with Task 1:
- `ColumnTransformer` → apply different preprocessing to numeric and text columns.
- `OneHotEncoder` → turns categories like "Petrol/Diesel/CNG" into 0/1 columns a model can use.
- Regression models and regression metrics (MAE, MSE, RMSE, R²).

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
os.makedirs("images", exist_ok=True)
os.makedirs("models", exist_ok=True)
print("pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)

## Step 2 — Load the dataset
Different versions of this file name some columns differently (`Kms_Driven` vs `Driven_kms`, `Seller_Type` vs `Selling_type`). We rename them to one standard so the notebook works with either version.

**Expected output:** the first rows, with columns `Car_Name, Year, Selling_Price, Present_Price, Driven_kms, Fuel_Type, Selling_type, Transmission, Owner`.

In [ ]:
CANDIDATES = ["car data.csv", "car_data.csv", os.path.join("data", "car data.csv"), os.path.join("data", "car_data.csv")]
csv_path = next((p for p in CANDIDATES if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(
        "'car data.csv' not found. Download it from "
        "https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho "
        "and upload it next to this notebook (or into a 'data' folder)."
    )

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
df = df.rename(columns={"Kms_Driven": "Driven_kms", "Seller_Type": "Selling_type"})

REQUIRED = ["Car_Name", "Year", "Selling_Price", "Present_Price", "Driven_kms",
            "Fuel_Type", "Selling_type", "Transmission", "Owner"]
missing_cols = [c for c in REQUIRED if c not in df.columns]
assert not missing_cols, f"Missing expected columns: {missing_cols}. Found: {list(df.columns)}"

print("Loaded:", csv_path, "| shape:", df.shape)
df.head()

## Step 3 — Understand the dataset
| Column | Meaning | Type |
|---|---|---|
| `Car_Name` | Model name (e.g. "city", "ritz") | text (many unique values) |
| `Year` | Year the car was bought | number |
| **`Selling_Price`** | **Target** — price the owner is asking (lakhs ₹) | number |
| `Present_Price` | Current ex-showroom price of the same car **new** (lakhs ₹) | number |
| `Driven_kms` | Kilometres driven | number |
| `Fuel_Type` | Petrol / Diesel / CNG | category |
| `Selling_type` | Dealer or Individual seller | category |
| `Transmission` | Manual / Automatic | category |
| `Owner` | Number of previous owners | number |

**About the PDF's example features:** the task mentions "brand goodwill, horsepower, mileage". This dataset has no horsepower column. `Present_Price` acts as a stand-in for brand/segment value (a premium brand has a higher showroom price), and `Driven_kms` is the usage "mileage". Mention this honestly in your README and video.

In [ ]:
df.info()
print("\nUnique values per column:\n", df.nunique())
df.describe().round(2)

## Step 4 — Data cleaning: missing values, duplicates, invalid values
**Expected output:** usually 0 missing values and a couple of duplicate rows (removed).

In [ ]:
print("Missing values:\n", df.isnull().sum())
for col in ["Year", "Selling_Price", "Present_Price", "Driven_kms", "Owner"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in ["Car_Name", "Fuel_Type", "Selling_type", "Transmission"]:
    df[col] = df[col].astype(str).str.strip()

rows_before = len(df)
df = df.dropna(subset=["Year", "Selling_Price", "Present_Price", "Driven_kms"])
n_dupes = df.duplicated().sum()
df = df.drop_duplicates()
df = df[(df["Selling_Price"] > 0) & (df["Present_Price"] > 0) & (df["Driven_kms"] >= 0)].reset_index(drop=True)
print(f"\nRows before: {rows_before} | duplicates removed: {n_dupes} | rows after: {len(df)}")

## Step 5 — Feature engineering
- **`Car_Age`** = (newest year in the data + 1) − `Year`. Age is more meaningful than the raw year. We use the dataset's own newest year (not today's date) so results don't change depending on when you run the notebook.
- **Drop `Car_Name`:** it has ~98 different values for ~300 cars — too many for so little data (the model would memorise names). The names are model names, not clean brands, so extracting a reliable brand isn't possible here.
- **Fill missing `Owner`** (if any) with 0 — the most common value.

In [ ]:
REFERENCE_YEAR = int(df["Year"].max()) + 1
df["Car_Age"] = REFERENCE_YEAR - df["Year"]
df["Owner"] = df["Owner"].fillna(0)
print("Reference year:", REFERENCE_YEAR)
print("Car_Name unique values:", df["Car_Name"].nunique())

NUM_FEATURES = ["Present_Price", "Driven_kms", "Car_Age", "Owner"]
CAT_FEATURES = ["Fuel_Type", "Selling_type", "Transmission"]
TARGET = "Selling_Price"
df[NUM_FEATURES + CAT_FEATURES + [TARGET]].head()

## Step 6 — Exploratory Data Analysis
### 6a. Distribution of the target
**What to look for:** most cars are cheap with a long tail of expensive ones (**right-skewed**). Skew means a few expensive cars can dominate error metrics — keep that in mind when reading results.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Selling price (lakhs ₹)")
sns.histplot(np.log1p(df[TARGET]), kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("log(1 + selling price) — less skewed")
plt.tight_layout()
plt.savefig("images/01_price_distribution.png", dpi=120)
plt.show()
print("Skewness of Selling_Price:", round(df[TARGET].skew(), 2))

### 6b. Numeric features vs price
**What to look for:** a strong upward pattern for `Present_Price` (expensive new → expensive used), a downward pattern for `Car_Age` (depreciation), and a weaker, noisier pattern for `Driven_kms`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Present_Price", "Car_Age", "Driven_kms"]):
    sns.scatterplot(data=df, x=col, y=TARGET, hue="Transmission", alpha=0.7, ax=ax)
    ax.set_title(f"{col} vs Selling_Price")
plt.tight_layout()
plt.savefig("images/02_numeric_vs_price.png", dpi=120)
plt.show()

### 6c. Categorical features vs price

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, CAT_FEATURES):
    sns.boxplot(data=df, x=col, y=TARGET, hue=col, palette="Set2", legend=False, ax=ax)
    ax.set_title(f"Selling price by {col}")
plt.tight_layout()
plt.savefig("images/03_categorical_vs_price.png", dpi=120)
plt.show()
for col in CAT_FEATURES:
    print(df.groupby(col)[TARGET].agg(["count", "mean", "median"]).round(2), "\n")

### 6d. Correlation heatmap (numeric columns)
Remember: correlation describes a **linear association**, not cause-and-effect.

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df[NUM_FEATURES + [TARGET]].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.tight_layout()
plt.savefig("images/04_correlation.png", dpi=120)
plt.show()

## Step 7 — Train / test split
80% training, 20% testing. The test set stays locked away until the very end.

**Is `Present_Price` data leakage?** No — leakage means using information you wouldn't have at prediction time. The new-car showroom price is publicly known *before* a used car is sold, so using it is legitimate. It is simply a very strong predictor.

In [ ]:
X = df[NUM_FEATURES + CAT_FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## Step 8 — Preprocessing pipeline
- **Numeric columns → `StandardScaler`** (important for linear models, harmless for trees).
- **Categorical columns → `OneHotEncoder`**: `Fuel_Type = Diesel` becomes `Fuel_Type_Diesel = 1, Fuel_Type_Petrol = 0, ...`. `handle_unknown="ignore"` prevents crashes if a new category appears later.
- Wrapping everything in a `Pipeline` means preprocessing is learned **only from training data** in every step → no leakage.

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
])

## Step 9 — Model selection and comparison
| Model | Idea |
|---|---|
| Linear Regression | price = weighted sum of features (a straight-line relationship) |
| Ridge Regression | linear regression with a penalty that prevents extreme weights |
| Decision Tree | splits cars into groups by yes/no rules; can overfit |
| Random Forest | averages many trees → more stable |
| Gradient Boosting | builds trees one after another, each fixing the previous errors |

We compare them with **5-fold cross-validation on the training set** using R² and RMSE.

**Metrics in plain English:**
- **MAE** (Mean Absolute Error): average size of the error, in lakhs. MAE = 0.5 → off by ₹50,000 on average.
- **MSE** (Mean Squared Error): average of squared errors; punishes big mistakes heavily (units: lakhs²).
- **RMSE** = √MSE: back in lakhs; larger than MAE when there are some big misses.
- **R²**: share of price variation explained by the model. 1.0 = perfect, 0 = no better than always guessing the average price.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
                            scoring={"r2": "r2", "rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error"})
    rows.append({
        "Model": name,
        "CV R² (mean)": scores["test_r2"].mean(),
        "CV R² (std)": scores["test_r2"].std(),
        "CV RMSE": -scores["test_rmse"].mean(),
        "CV MAE": -scores["test_mae"].mean(),
    })

cv_df = pd.DataFrame(rows).sort_values("CV R² (mean)", ascending=False).reset_index(drop=True)
cv_df.round(4)

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=cv_df, x="CV R² (mean)", y="Model", hue="Model", palette="viridis", legend=False)
plt.errorbar(cv_df["CV R² (mean)"], range(len(cv_df)), xerr=cv_df["CV R² (std)"], fmt="none", c="black")
plt.title("5-fold cross-validated R² (training data only)")
plt.tight_layout()
plt.savefig("images/05_model_comparison.png", dpi=120)
plt.show()

## Step 10 — Train the best model on the full training set

In [ ]:
best_name = cv_df.loc[0, "Model"]
best_model = Pipeline([("prep", preprocessor), ("model", models[best_name])])
best_model.fit(X_train, y_train)
print("Selected model:", best_name)

## Step 11 — Predict and evaluate on the unseen test set
We also evaluate on the training set: if training R² is much higher than test R², the model is **overfitting** (memorising instead of learning).

In [ ]:
def regression_report(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse,
            "RMSE": np.sqrt(mse), "R²": r2_score(y_true, y_pred)}

y_pred_test = best_model.predict(X_test)
y_pred_train = best_model.predict(X_train)

results = pd.DataFrame({
    "Train": regression_report(y_train, y_pred_train),
    "Test": regression_report(y_test, y_pred_test),
}).round(4)
print(f"Model: {best_name}  (prices in lakhs ₹)")
results

### Compare ALL models on the test set (for your report)
Selection was already made using cross-validation; this table is for transparency only. Choosing the model by test score would be a subtle form of leakage.

In [ ]:
test_rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)]).fit(X_train, y_train)
    test_rows.append({"Model": name, **regression_report(y_test, pipe.predict(X_test))})
pd.DataFrame(test_rows).sort_values("R²", ascending=False).round(4)

## Step 12 — Visualise results
### 12a. Actual vs predicted
**How to read:** each dot is a test car. Dots on the dashed diagonal = perfect predictions. Dots far from the line = big errors (often the few expensive cars).

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_test, alpha=0.7, edgecolor="k")
lims = [0, max(y_test.max(), y_pred_test.max()) * 1.05]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlim(lims); plt.ylim(lims)
plt.xlabel("Actual selling price (lakhs ₹)")
plt.ylabel("Predicted selling price (lakhs ₹)")
plt.title(f"Actual vs Predicted — {best_name}")
plt.legend()
plt.tight_layout()
plt.savefig("images/06_actual_vs_predicted.png", dpi=120)
plt.show()

### 12b. Residuals (errors)
Residual = actual − predicted. Ideally residuals are centred around 0 with no pattern. A funnel shape (errors grow with price) means the model is less reliable for expensive cars.

In [ ]:
residuals = y_test - y_pred_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_pred_test, residuals, alpha=0.7, edgecolor="k")
axes[0].axhline(0, color="red", ls="--")
axes[0].set_xlabel("Predicted price"); axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs predicted")
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title("Distribution of residuals")
plt.tight_layout()
plt.savefig("images/07_residuals.png", dpi=120)
plt.show()

## Step 13 — Which factors are associated with price?
**Permutation importance:** shuffle one feature in the test set and measure how much R² drops. A big drop → the model relies heavily on that feature. It works for any model and is computed on the original (un-encoded) columns, so it's easy to read.

⚠️ Importance shows what the **model uses**, not what *causes* prices in the real world.

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=20, random_state=RANDOM_STATE, scoring="r2")
imp = pd.DataFrame({"feature": X_test.columns, "importance": perm.importances_mean, "std": perm.importances_std})
imp = imp.sort_values("importance", ascending=True)

plt.figure(figsize=(7, 4))
plt.barh(imp["feature"], imp["importance"], xerr=imp["std"], color="teal")
plt.xlabel("Mean drop in R² when shuffled")
plt.title(f"Permutation importance — {best_name}")
plt.tight_layout()
plt.savefig("images/08_feature_importance.png", dpi=120)
plt.show()
imp.sort_values("importance", ascending=False).round(4)

### Linear-model coefficients (for direction of effect)
The linear model's coefficients show the **direction**: positive = associated with a higher price, negative = lower price (holding other features constant). Numeric features are standardised, so their coefficients mean "change in price (lakhs) for +1 standard deviation".

In [ ]:
lin = Pipeline([("prep", preprocessor), ("model", LinearRegression())]).fit(X_train, y_train)
feature_names = lin.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lin.named_steps["model"].coef_, index=feature_names).sort_values()
plt.figure(figsize=(7, 5))
coefs.plot(kind="barh", color=["firebrick" if c < 0 else "seagreen" for c in coefs])
plt.title("Linear Regression coefficients")
plt.tight_layout()
plt.savefig("images/09_linear_coefficients.png", dpi=120)
plt.show()

## Step 14 — Save the model and predict a new car

In [ ]:
MODEL_PATH = "models/car_price_model.joblib"
joblib.dump({"model": best_model, "reference_year": REFERENCE_YEAR,
             "num_features": NUM_FEATURES, "cat_features": CAT_FEATURES}, MODEL_PATH)
print("Saved to", MODEL_PATH)

In [ ]:
bundle = joblib.load(MODEL_PATH)

def predict_price(year, present_price, driven_kms, fuel_type, selling_type, transmission, owner=0):
    """Predict selling price (lakhs ₹) for one car."""
    car = pd.DataFrame([{
        "Present_Price": present_price, "Driven_kms": driven_kms,
        "Car_Age": bundle["reference_year"] - year, "Owner": owner,
        "Fuel_Type": fuel_type, "Selling_type": selling_type, "Transmission": transmission,
    }])
    return float(bundle["model"].predict(car[bundle["num_features"] + bundle["cat_features"]])[0])

example = predict_price(year=2014, present_price=9.5, driven_kms=40000,
                        fuel_type="Diesel", selling_type="Dealer", transmission="Manual")
print(f"Predicted selling price: {example:.2f} lakhs ₹")

## Step 15 — Results summary for the README (from YOUR run)

In [ ]:
summary = results["Test"].to_frame("Test value")
summary.loc["Best model"] = best_name
summary.to_csv("images/results_summary.csv")
summary

## Step 16 — Conclusions, limitations and improvements
**Conclusions (check them against your output):**
- Tree-based ensembles usually capture the non-linear price patterns better than linear models on this data — confirm with your CV table.
- `Present_Price` (value of the car when new) and `Car_Age` are typically the features the model relies on most; kilometres and fuel type add smaller amounts.

**Limitations:**
- Small dataset (~300 cars) → metrics vary with the random split; a handful of expensive cars strongly affect RMSE.
- No horsepower, engine size, condition, city or brand columns.
- Prices are asking prices from one website at one point in time; they may not generalise to other markets or years.

**Improvements:** try predicting `log(price)` to reduce the effect of skew; tune hyper-parameters with `GridSearchCV`; use the larger `Car details v3.csv` file in the same Kaggle dataset (it includes engine, max power, mileage) for richer features; deploy with Streamlit.